In [19]:
import torch
import torch.nn as nn
from torch.optim import SGD
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
import torchvision
import numpy as np
import matplotlib.pyplot as plt
import cv2
import random

# Carregando a Rede

In [20]:
class MyCNN(nn.Module):
    def __init__(self):
        super().__init__()
        self.conv1 = nn.Conv2d(1,6,3,padding=1)
        self.pool = nn.MaxPool2d(2,2)
        self.conv2 = nn.Conv2d(6,18,3)
        self.fc1 = nn.Linear(18*6*6,100)
        self.fc2 = nn.Linear(100,50)
        self.fc3 = nn.Linear(50,6)
        self.R = nn.ReLU()
    def forward(self,x):
        x = self.pool(self.R(self.conv1(x)))
        x = self.pool(self.R(self.conv2(x)))
        x = x.view(-1,18*6*6)
        x = self.R(self.fc1(x))
        x = self.R(self.fc2(x))
        x = self.fc3(x)
        return x.squeeze()

In [21]:
RedeNeural = MyCNN()

In [22]:
RedeNeural.load_state_dict(torch.load('CNN4.pth'))

C:\Users\R2\AppData\Local\Temp\ipykernel_2836\4288584823.py:1: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  RedeNeural.load_state_dict(torch.load('CNN4.pth'))


<All keys matched successfully>

In [23]:
print(RedeNeural)

MyCNN(
  (conv1): Conv2d(1, 6, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
  (pool): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
  (conv2): Conv2d(6, 18, kernel_size=(3, 3), stride=(1, 1))
  (fc1): Linear(in_features=648, out_features=100, bias=True)
  (fc2): Linear(in_features=100, out_features=50, bias=True)
  (fc3): Linear(in_features=50, out_features=6, bias=True)
  (R): ReLU()
)


# Parte do desenho

In [24]:
def ImgNormalizer(img):
    w = len(img[0])
    h = len(img[1])
    DrawX = []
    DrawY = []
    for x in range(w):
        for y in range(h):
            if img[x][y] == 255:
                DrawX.append(x)
                DrawY.append(y)

    # Se não ouver desenho nenhum, retorna imagens completamente vazias
    if not DrawX:
        imgSmall = np.full((28,28), 0, dtype=np.uint8)
        imgSmall2 = np.full((28,28), 0, dtype=np.uint8)
        return imgSmall,imgSmall2
        
    # Pega os valores mínimos e máximos das coordenadas
    MaxXValue = np.max(DrawX)
    MinXValue = np.min(DrawX)
    MaxYValue = np.max(DrawY)
    MinYValue = np.min(DrawY)
    
    # Calcula as amplitudes
    XAmplitude = MaxXValue - MinXValue
    YAmplitude = MaxYValue - MinYValue
    
    if XAmplitude > YAmplitude:
        # Altera as coordenadas para ir de 0 ate a aplitude maxima e centraliza a amplitude menor
        Amp = XAmplitude
        for i in range(len(DrawX)):
            DrawX[i] = DrawX[i] - MinXValue
            DrawY[i] = DrawY[i] - MinYValue + (Amp/2) - (YAmplitude/2)
    else:
        # Altera as coordenadas para ir de 0 ate a aplitude maxima e centraliza a amplitude menor
        Amp = YAmplitude
        for i in range(len(DrawX)):
            DrawX[i] = DrawX[i] - MinXValue + (Amp/2) - (XAmplitude/2)
            DrawY[i] = DrawY[i] - MinYValue
    
    DrawSmall = []
    for i in range(len(DrawX)):
        # Reduz a amplitude para 28x28 com 2 pixels de borda e arredonda os resultados
        newpair = []
        newpair.append(((23/Amp)*DrawX[i]) + 2)
        newpair.append(((23/Amp)*DrawY[i]) + 2) 
        newpairR = np.int32(np.rint(newpair))
        DrawSmall.append(newpairR)
    # Deixa apenas os pares únicos, excluindo os repetidos após o arredondamento
    DrawSmall = np.unique(DrawSmall,axis=0)

    #Desenha a imagem 28x28
    imgSmall = np.full((28,28), 0, dtype=np.uint8)
    imgSmall2 = np.full((28,28), 0, dtype=np.uint8)
    for pair in DrawSmall:
        imgSmall[pair[0]][pair[1]] = 255
        cv2.circle(imgSmall2,(pair[1],pair[0]),1,(255,255,255),-1)

    return imgSmall,imgSmall2

In [25]:
class DrawingApp:
    def __init__(self):
        # Variáveis iniciais
        self.isDrawing = False
        self.Acertou = False
    
    def drawLine(self,event,x,y,flags,param):
        #Desenha na imagem enquanto o mouse se move após o botão esquerdo do mouse é pressionado
        #Desenha em 2 imagens ao mesmo tempo: Uma para ter uma referencia do desenho e uma para gerar as coordenadas do mouse
        img1 = param[0]
        img2 = param[1]
        if event == cv2.EVENT_LBUTTONDOWN:
            self.isDrawing = True
        elif event == cv2.EVENT_MOUSEMOVE and self.isDrawing:
            cv2.circle(img1,(x,y),5,(255,255,255),-1)
            cv2.circle(img2,(x,y),0,(255,255,255),-1)
        elif event == cv2.EVENT_LBUTTONUP:
            self.isDrawing = False

    def run(self):

        # Cria imagens base vazias
        img = np.full((640,640), 0, dtype=np.uint8)
        img2 = np.full((640,640), 0, dtype=np.uint8)

        countImg = np.full((800,800), 0, dtype=np.uint8)
        font = cv2.FONT_HERSHEY_SIMPLEX
        texto1 = "Palpite:"
        cv2.putText(countImg, texto1, (20, 50), font, 1, (255,255,255), 3, cv2.LINE_AA)

        # Cria um vetor com as duas imagens
        images = [img,img2]

        #gera o nome da janela
        windowName = 'drawing app'
        cv2.namedWindow(windowName)
        windowName2 = 'count'
        cv2.namedWindow(windowName2)

        #chama a função de desenho que chama a função drawLine com os eventos do mouse
        cv2.setMouseCallback(windowName,self.drawLine,images)

        while True:
            cv2.imshow(windowName,images[0])
            cv2.imshow(windowName2,countImg)
            wait = cv2.waitKey(1)
            if wait == ord('q'):
                # Sai do Loop da etapa de desenho
                break
            elif wait == ord('r'):
                # Limpa o conteudo da tela para que novas imagens possam ser geradas
                images[0] = np.full((640,640), 0, dtype=np.uint8)
                images[1] = np.full((640,640), 0, dtype=np.uint8)
            elif wait == ord('s'):
                # Gera as imagens na resolução 28x28
                imagesSmall = ImgNormalizer(images[1])
                
                SoftMax = nn.Softmax(dim=0)
                prediction = RedeNeural((torch.from_numpy(np.asarray(imagesSmall[0]))/255).view(1,28,28))
                prediction_answer = prediction.argmax()
                prediction_prob = SoftMax(prediction)

                

                print(prediction)
                print(prediction[prediction_answer])
                print(prediction_answer)
                print(prediction_prob)

                prediction_prob_formated = "{:.2f}".format(prediction_prob[prediction_answer].item()*100)
                texto2 = "Confianca: " + str(prediction_prob_formated) + "%"

                prediction_answer_formated = "{:.2f}".format(prediction[prediction_answer])
                texto3 = "Valor: " + str(prediction_answer_formated)

                answers_list = ("Baixo","Cima","Esquerda","Direita","Circulo","Coroa")
                guess = answers_list[prediction_answer]

                textoBaixo = "Baixo: " + str("{:.2f}".format(prediction[0]))
                textoCima = "Cima: " + str("{:.2f}".format(prediction[1]))
                textoEsquerda = "Esquerda: " + str("{:.2f}".format(prediction[2]))
                textoDireita = "Direita: " + str("{:.2f}".format(prediction[3]))
                textoCirculo = "Circulo: " + str("{:.2f}".format(prediction[4]))
                textoCoroa = "Coroa: " + str("{:.2f}".format(prediction[5]))

                
                countImg = np.full((800,800), 0, dtype=np.uint8)
                cv2.putText(countImg, texto1, (20, 50), font, 1, (255,255,255), 3, cv2.LINE_AA)

                cv2.putText(countImg, guess, (20, 100), font, 1, (255,255,255), 3, cv2.LINE_AA)

                cv2.putText(countImg, texto2, (20, 150), font, 1, (255,255,255), 3, cv2.LINE_AA)

                cv2.putText(countImg, textoBaixo, (20, 200), font, 1, (255,255,255), 3, cv2.LINE_AA)

                cv2.putText(countImg, textoCima, (20, 250), font, 1, (255,255,255), 3, cv2.LINE_AA)

                cv2.putText(countImg, textoEsquerda, (20, 300), font, 1, (255,255,255), 3, cv2.LINE_AA)

                cv2.putText(countImg, textoDireita, (20, 350), font, 1, (255,255,255), 3, cv2.LINE_AA)

                cv2.putText(countImg, textoCirculo, (20, 400), font, 1, (255,255,255), 3, cv2.LINE_AA)

                cv2.putText(countImg, textoCoroa, (20, 450), font, 1, (255,255,255), 3, cv2.LINE_AA)
        
        # Fecha a janela e termina processo
        cv2.destroyAllWindows()

In [26]:
def Drawing():
    app = DrawingApp()
    app.run()

In [27]:
if __name__ == '__main__':
    Drawing()

tensor([-12.3113,   2.0521,   0.4512,  -5.0289,  -0.4414,  18.4687],
       grad_fn=<SqueezeBackward0>)
tensor(18.4687, grad_fn=<SelectBackward0>)
tensor(5)
tensor([4.2893e-14, 7.4187e-08, 1.4964e-08, 6.2387e-11, 6.1293e-09, 1.0000e+00],
       grad_fn=<SoftmaxBackward0>)
tensor([-11.9965,   1.1062,   0.6027,  -4.6115,  -0.2660,  18.1147],
       grad_fn=<SqueezeBackward0>)
tensor(18.1147, grad_fn=<SelectBackward0>)
tensor(5)
tensor([8.3732e-14, 4.1050e-08, 2.4812e-08, 1.3494e-10, 1.0408e-08, 1.0000e+00],
       grad_fn=<SoftmaxBackward0>)
tensor([-4.9280, 17.5384, -5.7715, -0.4318, -8.7650,  4.3563],
       grad_fn=<SqueezeBackward0>)
tensor(17.5384, grad_fn=<SelectBackward0>)
tensor(1)
tensor([1.7497e-10, 1.0000e+00, 7.5268e-11, 1.5691e-08, 3.7719e-12, 1.8840e-06],
       grad_fn=<SoftmaxBackward0>)
